# 10 — Fairness Audit

We don't claim to "solve" fairness. We measure whether model performance and — more decision-relevant — who actually gets selected for a capacity-limited intervention differ across demographic subgroups, and how that trade-off moves as the threshold changes. `race`/`gender`/`age` are audit dimensions only; they were never model input features (see `src/models.py` — not in `NUMERIC_FEATURES` or `CATEGORICAL_FEATURES`).

In [1]:
import sys
sys.path.insert(0, '../src')
import joblib
import pandas as pd
from fairness import subgroup_metrics, allocation_rates_by_group
from optimization import expected_net_benefit, select_top_k_greedy

pipe = joblib.load('../data/processed/models/xgboost.joblib')
X_train, X_test, y_train, y_test = joblib.load('../data/processed/models/splits.joblib')
df = pd.read_csv('../data/processed/diabetic_data_features.csv')
demo = df.loc[X_test.index, ['race', 'gender', 'age']]

risk = pipe.predict_proba(X_test)[:, 1]
y_true = y_test.values

## Model performance by subgroup, at a 0.5 threshold

In [2]:
subgroup_metrics(y_true, risk, demo['race'])

,group,n,base_rate,mean_predicted_risk,selection_rate,recall,false_negative_rate,precision
0,AfricanAmerican,3814,0.109,0.441,0.344,0.530,0.470,0.168
1,Asian,134,0.097,0.398,0.201,0.615,0.385,0.296
2,Caucasian,15011,0.119,0.448,0.344,0.558,0.442,0.192
3,Hispanic,387,0.078,0.401,0.274,0.567,0.433,0.160
4,Other,313,0.096,0.408,0.265,0.400,0.600,0.145
5,Unknown,416,0.091,0.374,0.190,0.447,0.553,0.215


In [3]:
subgroup_metrics(y_true, risk, demo['gender'])

,group,n,base_rate,mean_predicted_risk,selection_rate,recall,false_negative_rate,precision
0,Female,10657,0.111,0.447,0.351,0.563,0.437,0.179
1,Male,9417,0.119,0.438,0.322,0.534,0.466,0.197


In [4]:
subgroup_metrics(y_true, risk, demo['age'])

,group,n,base_rate,mean_predicted_risk,selection_rate,recall,false_negative_rate,precision
0,[0-10),36,0.028,0.092,0.000,0.000,1.000,NaN
1,[10-20),116,0.043,0.282,0.138,0.400,0.600,0.125
2,[20-30),308,0.117,0.412,0.305,0.722,0.278,0.277
3,[30-40),724,0.124,0.402,0.268,0.544,0.456,0.253
4,[40-50),1998,0.102,0.407,0.277,0.586,0.414,0.215
5,[50-60),3443,0.100,0.403,0.247,0.493,0.507,0.200
6,[60-70),4437,0.119,0.439,0.302,0.506,0.494,0.200
7,[70-80),5222,0.122,0.470,0.390,0.564,0.436,0.176
8,[80-90),3307,0.122,0.488,0.458,0.615,0.385,0.165
9,[90-100),484,0.114,0.467,0.362,0.455,0.545,0.143


**Reading this:** recall (share of actual readmissions the model catches) is meaningfully lower for `AfricanAmerican` (≈0.53) than `Caucasian` (≈0.56) patients, with a correspondingly higher false-negative rate — a real, if moderate, gap. `Asian`/`Hispanic`/`Other`/`Unknown` race groups and the youngest age bands have small subgroup sizes (well under 500 rows for most), so their metrics are noisy and shouldn't be over-interpreted at face value — flagged explicitly rather than silently treated as equally reliable as the large-n groups.

## The metric that actually matters once optimization enters the picture

Subgroup recall at a fixed threshold doesn't capture what happens once only `capacity` patients can be selected — that's a ranking/allocation problem, not a threshold problem. Allocation rate by group under the capacity-500 utility-optimized policy from `09_optimization.ipynb`:

In [5]:
net_benefit = expected_net_benefit(risk, 0.20, 10_000, 100)
selected = select_top_k_greedy(net_benefit, capacity=500)
allocation_rates_by_group(selected, demo['race'])

,group,n,n_selected,selection_rate
0,AfricanAmerican,3814,88,0.023
1,Asian,134,0,0.000
2,Caucasian,15011,409,0.027
3,Hispanic,387,1,0.003
4,Other,313,2,0.006
5,Unknown,416,0,0.000


Caucasian and African American patients are selected at broadly similar rates (~2.3-2.7%) once weighted by group size — the disparity that matters more here is *which* African American patients get missed (lower recall = more false negatives, i.e. genuinely at-risk patients who don't get flagged), not an under-allocation of intervention slots relative to population share.

## Threshold trade-off

Whether a lower classification threshold narrows or widens the recall gap between groups is a live empirical question, not something to assume either way — a lower threshold selects more patients from every group, which can narrow an FNR gap if the gap comes from a shared borderline-risk zone, or leave it unchanged if it comes from a genuinely different score distribution.

In [6]:
from fairness import threshold_fairness_sweep
sweep = threshold_fairness_sweep(y_true, risk, demo['race'], thresholds=(0.2, 0.3, 0.4, 0.5))
sweep[sweep['group'].isin(['Caucasian', 'AfricanAmerican'])].pivot(index='threshold', columns='group', values='recall')

group,AfricanAmerican,Caucasian
threshold,,
0.2,0.978,0.992
0.3,0.906,0.936
0.4,0.757,0.785
0.5,0.530,0.558


## Summary

- A real, moderate recall gap exists between African American and Caucasian patients (~3 points at the default threshold) — worth surfacing to a hospital deploying this, not something this model or dataset can fix on its own.
- Under the capacity-constrained utility-optimized allocation, selection rates by race track population share reasonably closely; the fairness concern here is concentrated in *who gets missed* (recall/FNR), not in raw allocation share.
- Several demographic subgroups are too small in this test set for their metrics to be trusted at face value — reported for completeness, flagged as low-confidence rather than omitted or presented with false precision.